## Naive Bayes Classifier

**Simon Grishin**

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
# Import data
train_data = pd.read_csv('fraudTrain.csv').drop(columns=['Unnamed: 0'])
test_data = pd.read_csv('fraudTest.csv').drop(columns=['Unnamed: 0'])

In [ ]:
train_data.head()

In [ ]:
train_data['merchant'].value_counts()

In [ ]:
# Target Variable Class imbalance
train_data['is_fraud'].value_counts()

In [ ]:
train_data.describe()

### Exploratory Data Analysis:

**Drop:**
* Transaction number
* cc_num
* first name
* last_name
* unix_time

**Transformed Variables:**
* Locational variables for merchant "merch_lat" and "merch_long"
* Locational variables for cardholders "lat" and "long"
* Convert dob to age in numerical value 
* Convert date time to month, day, year (numerical)

**Categorical Variables:**
* merchant
* category
* city
* state
* zip
* gender

**Numerical Variables:**
* city_pop
* amt

In [ ]:
# Data Preprocessing
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder
from datetime import datetime
import reverse_geocoder as rg

transformed_data = train_data.copy()

# Computee distance between merchant and customer
transformed_data['distance'] = np.sqrt((transformed_data['lat'] - transformed_data['merch_lat'])**2 + (transformed_data['long'] - transformed_data['merch_long'])**2)

# Transforming dob to age
transformed_data['dob'] = pd.to_datetime(transformed_data['dob'])
transformed_data['age'] = (datetime.now() - transformed_data['dob']).dt.days / 365
transformed_data.drop(columns=['dob'], inplace=True)

# Transforming date to day, month, year
date = transformed_data['trans_date_trans_time'].apply(lambda x: datetime.strptime(x, '%Y-%m-%d %H:%M:%S'))
transformed_data['hour'] = date.dt.hour
transformed_data['day'] = date.dt.dayofweek
transformed_data['month'] = date.dt.month
transformed_data.drop(columns=['trans_date_trans_time'], inplace=True)

# Log-transform skewed data
transformed_data['amt'] = np.log1p(transformed_data['amt'])

transformed_data.drop(columns=['first', 'last', 'street', 'lat', 'long', 'merch_lat', 'merch_long', 'cc_num', 'trans_num', 'unix_time', 'job'], inplace=True)

# Transforming categorical data (Label encoding)
label_encoder = LabelEncoder()
label_vars = ['gender']

for var in label_vars:
    transformed_data[var] = label_encoder.fit_transform(transformed_data[var])

# Transform categorical data (frequency encoding)
frequency_vars = ['merchant', 'zip', 'city', 'state', 'category']

for col in frequency_vars:
    freq_encoding = transformed_data[col].value_counts(normalize=True).to_dict()
    transformed_data[col] = transformed_data[col].map(freq_encoding)
    

### Test Data Preprocessing

In [ ]:
transformed_Y_data = train_data.copy()

# Computee distance between merchant and customer
transformed_Y_data['distance'] = np.sqrt((transformed_Y_data['lat'] - transformed_Y_data['merch_lat'])**2 + (transformed_Y_data['long'] - transformed_Y_data['merch_long'])**2)

# Transforming dob to age
transformed_Y_data['dob'] = pd.to_datetime(transformed_Y_data['dob'])
transformed_Y_data['age'] = (datetime.now() - transformed_Y_data['dob']).dt.days / 365
transformed_Y_data.drop(columns=['dob'], inplace=True)

# Transforming date to day, month, year
date = transformed_Y_data['trans_date_trans_time'].apply(lambda x: datetime.strptime(x, '%Y-%m-%d %H:%M:%S'))
transformed_Y_data['hour'] = date.dt.hour
transformed_Y_data['day'] = date.dt.dayofweek
transformed_Y_data['month'] = date.dt.month
transformed_Y_data.drop(columns=['trans_date_trans_time'], inplace=True)

# Log-transform skewed data
transformed_Y_data['amt'] = np.log1p(transformed_Y_data['amt'])

transformed_Y_data.drop(columns=['first', 'last', 'street', 'lat', 'long', 'merch_lat', 'merch_long', 'cc_num', 'trans_num', 'unix_time', 'job'], inplace=True)

# Transforming categorical data (Label encoding)
label_encoder = LabelEncoder()
label_vars = ['gender']

for var in label_vars:
    transformed_Y_data[var] = label_encoder.fit_transform(transformed_Y_data[var])

# Transform categorical data (frequency encoding)
frequency_vars = ['merchant','zip', 'city', 'state', 'category']

for col in frequency_vars:
    freq_encoding = transformed_Y_data[col].value_counts(normalize=True).to_dict()
    transformed_Y_data[col] = transformed_Y_data[col].map(freq_encoding)


In [ ]:
train_data.groupby('job')['is_fraud'].mean().sort_values(ascending=False).head(20).plot(kind='barh', figsize=(10, 6), color='orange')

## Feature Selection

**Feature Engineering:** 
* Different Encodings: Frequency, Target, One-Hot
* Log transformations
* Grouping certain features together (e.g. City and Zip code, Merchant frequency and distance)
* Hash encoding maps categories toa fixed number of numerical features
* Normalization for numerical features
* Temporal variables can be sine and cosine transformed to represent cyclical nature

**Encoding:**

* **Job:** A persons job probably doesn't have as much direct correlation but frequency would help distribute types of jobs (frequency)
* **Category:** Target encoding would provide a look into which categories are the most fraudulent (target)
* **Merchant:** Frequency encoding provides details on how often that merchant shows up. The less it shows up probably indicates more fraud (frequency)
* **City:** The frequency of fraud in a city is a good indicator of fraudulent transactions in cities (fraud)
* **State:** Since state is quite high-level, number of non-fraudulant transactions is a good measure of country wide distribution of clean transactions (target (1-fraud))
* **Zip:** Zip-code can show certain areas being more or less fraudulent (target)


**Feature Selection Techniques:**
* PCA to reduce dimensionality and engineering better features but only useful for very correlated features
* Recursive Feature Elimination (RFE): Iteratively trains the model, removing the least significant features at each step, and evaluates performance to identify the optimal feature subset.
* Genetic Algorithms (GA): Utilize evolutionary techniques to explore various feature combinations, optimizing for the best subset. Research has shown that combining GA with Naive Bayes can enhance anomaly detection performance. 


**Prediction Strategies:**
* Reducing False Positive and False Negatives

**Resources:**
* [Naive Bayes for Crime Predition](https://dl.acm.org/doi/fullHtml/10.1145/3488933.3488973)

In [ ]:
transformed_data

In [ ]:
# Covariance matrix
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(20, 20))
sns.heatmap(transformed_data.corr(), annot=True, fmt='.2f', cmap='coolwarm')
plt.show()

## Model Development

In [ ]:
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import confusion_matrix

def compute_utility(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred)
    TP, FN, FP = cm[1, 1], cm[1, 0], cm[0, 1]
    utility = TP * (50) - FN * (100) - FP * (5)
    return utility


def train_and_test_model(training_data):
    # Split the data into features and target variable
    X = training_data.drop(columns=['is_fraud'])
    y = training_data['is_fraud']
    
    # Split the data into training and validation sets
    X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
    
    # Handle class imbalance using SMOTE
    smote = SMOTE(sampling_strategy=0.1, random_state=42)
    X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)
    
    # Standardizing numerical features
    scaler = StandardScaler()
    X_train_sm = scaler.fit_transform(X_train_sm)
    X_val = scaler.transform(X_val)
    
    # Train Naive Bayes Classifier
    gnb = GaussianNB()
    gnb.fit(X_train_sm, y_train_sm)
    
    # Prediction probabilities
    y_prob = gnb.predict_proba(X_val)[:, 1]

    # Set a threshold for classification
    threshold = 0.43565

    y_pred = (y_prob >= threshold).astype(int)

    # # Predictions
    # y_pred = gnb.predict(X_val)
    
    return y_val, y_pred

# Train and test the model
y_val, y_pred = train_and_test_model(transformed_data)

# Evaluate model performance
utility_score = compute_utility(y_val, y_pred)
print("Validation Utility Score:", utility_score)

In [ ]:
# Confusion matrix
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix
import seaborn as sns


cm = confusion_matrix(y_val, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Not Fraud', 'Fraud'], yticklabels=['Not Fraud', 'Fraud'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()


## Utility Score Test Set

In [ ]:
# Train model on full training data and predict on test data
X = transformed_data.drop(columns=['is_fraud'])
y = transformed_data['is_fraud']
X_test = transformed_Y_data.drop(columns=['is_fraud'])
y_test = transformed_Y_data['is_fraud']

# Handle class imbalance using SMOTE
smote = SMOTE(sampling_strategy=0.1, random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X, y)

# Standardizing numerical features
scaler = StandardScaler()
X_train_sm = scaler.fit_transform(X_train_sm)
X_test = scaler.transform(X_test)

# Train Naive Bayes Classifier
gnb = GaussianNB()
gnb.fit(X_train_sm, y_train_sm)

# Prediction probabilities
y_prob = gnb.predict_proba(X_test)[:, 1]

# Set a threshold for classification
threshold = 0.43565

y_pred = (y_prob >= threshold).astype(int)

# Evaluate model performance
utility_score = compute_utility(y_test, y_pred)
print("Test Utility Score:", utility_score)

## Parameter Estimation and Subset Selection

* Backward Stepwise Subset Selection
* Recursive Feature Elimination

In [ ]:
def backward_stepwise_selection(data, target_column, utility_function):
    remaining_features = list(data.columns)
    remaining_features.remove(target_column)
    best_features = remaining_features.copy()
    best_utility = float('-inf')
    
    while len(remaining_features) > 0:
        utilities = {}
        for feature in remaining_features:
            temp_features = [f for f in remaining_features if f != feature]
            temp_data = data[temp_features + [target_column]]
            y_val, y_pred = train_and_test_model(temp_data)
            utility = utility_function(y_val, y_pred)
            utilities[feature] = utility
        
        # Find the feature whose removal minimizes the F statistic (maximizes utility)
        worst_feature = min(utilities, key=utilities.get)
        worst_utility = utilities[worst_feature]
        
        if worst_utility > best_utility:
            best_utility = worst_utility
            best_features = [f for f in remaining_features if f != worst_feature]
        
        remaining_features.remove(worst_feature)
    
    return best_features, best_utility

# Perform backward stepwise subset selection
selected_features, max_utility = backward_stepwise_selection(transformed_data, 'is_fraud', compute_utility)
print("Selected Features:", selected_features)
print("Maximum Utility Score:", max_utility)